https://stackoverflow.com/questions/53514495/what-does-batch-repeat-and-shuffle-do-with-tensorflow-dataset

In [74]:
import tensorflow as tf

<h3 style='color:purple'>Create tf dataset from a list</h3>

In [75]:
daily_sales_numbers = [21, 22, -108, 31, -1, 32, 34,31]
tf_dataset = tf.data.Dataset.from_tensor_slices(daily_sales_numbers)
tf_dataset

<_TensorSliceDataset element_spec=TensorSpec(shape=(), dtype=tf.int32, name=None)>

<h3 style='color:purple'>Iterate through tf dataset</h3>

In [76]:
for sales in tf_dataset:
    print(sales.numpy())

21
22
-108
31
-1
32
34
31


<h3 style='color:purple'>Iterate through elements as numpy elements</h3>

In [77]:
for sales in tf_dataset.as_numpy_iterator():
    print(sales)

21
22
-108
31
-1
32
34
31


<h3 style='color:purple'>Iterate through first n elements in tf dataset</h3>

In [78]:
for sales in tf_dataset.take(3):
    print(sales.numpy())

21
22
-108


<h3 style='color:purple'>Filter sales numbers that are < 0</h3>

In [79]:
tf_dataset = tf_dataset.filter(lambda x: x>0)
for sales in tf_dataset.as_numpy_iterator():
    print(sales)

21
22
31
32
34
31


<h3 style='color:purple'>Convert sales numbers from USA dollars ($) to Indian Rupees (INR) Assuming 1->72 conversation rate</h3>

In [80]:
tf_dataset = tf_dataset.map(lambda x: x*72)
for sales in tf_dataset.as_numpy_iterator():
    print(sales)

1512
1584
2232
2304
2448
2232


<h3 style='color:purple'>Shuffe</h3>

In [81]:
tf_dataset = tf_dataset.shuffle(2)
for sales in tf_dataset.as_numpy_iterator():
    print(sales)

1584
1512
2304
2448
2232
2232


<h3 style='color:purple'>Batching</h3>

In [82]:
for sales_batch in tf_dataset.batch(2):
    print(sales_batch.numpy())

[1584 2232]
[2304 1512]
[2232 2448]


<h3 style='color:purple'>Perform all of the above operations in one shot</h3>

In [83]:
tf_dataset = tf.data.Dataset.from_tensor_slices(daily_sales_numbers)

tf_dataset = tf_dataset.filter(lambda x: x>0).map(lambda y: y*72).shuffle(2).batch(2)
for sales in tf_dataset.as_numpy_iterator():
    print(sales)

[1512 2232]
[2304 1584]
[2232 2448]


<h3>Images</h3>

In [84]:
images_ds = tf.data.Dataset.list_files('images/*/*', shuffle=False)

In [85]:
image_count = len(images_ds)
image_count

130

In [86]:
type(images_ds)

tensorflow.python.data.ops.from_tensor_slices_op._TensorSliceDataset

In [87]:
for file in images_ds.take(3):
    print(file.numpy())

b'images\\cat\\20 Reasons Why Cats Make the Best Pets....webp'
b'images\\cat\\7 Foods Your Cat Can_t Eat.webp'
b'images\\cat\\A cat appears to have caught the....webp'


In [88]:
images_ds = images_ds.shuffle(200)
for file in images_ds.take(3):
    print(file.numpy())

b'images\\dog\\Calculate Your Dog_s Age With This New....webp'
b'images\\cat\\Soon_ the internet will make its own....webp'
b'images\\dog\\The History of Dogs as Pets - ABC News.webp'


In [89]:
class_names = ["cat","dog"]

In [90]:
train_size = int(image_count*0.8)
train_ds = images_ds.take(train_size)
test_ds = images_ds.skip(train_size)

In [91]:
len(train_ds)

104

In [92]:
len(test_ds)

26

In [93]:
def get_label(file_path):
    import os
    return tf.strings.split(file_path, os.path.sep)[-2]

In [94]:
get_label("images\\dog\\20 Reasons Why Cats Make the Best Pets....webp")

<tf.Tensor: shape=(), dtype=string, numpy=b'dog'>

In [95]:
def process_image(file_path):
    label = get_label(file_path)
    img = tf.io.read_file(file_path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [128, 128])
    return img, label

In [96]:
# for myself
def my_func(x):
    y = tf.io.read_file(x)
    y = tf.image.decode_jpeg(y)
    return y

my_func("images\\cat\\20 Reasons Why Cats Make the Best Pets....webp")

<tf.Tensor: shape=(1, 641, 640, 3), dtype=uint8, numpy=
array([[[[157, 186, 217],
         [157, 186, 217],
         [157, 186, 217],
         ...,
         [ 94,  84,  76],
         [ 94,  84,  76],
         [ 94,  84,  76]],

        [[157, 186, 217],
         [157, 186, 217],
         [157, 186, 217],
         ...,
         [ 94,  84,  76],
         [ 94,  84,  76],
         [ 94,  84,  76]],

        [[157, 186, 217],
         [157, 186, 217],
         [157, 186, 217],
         ...,
         [ 94,  84,  76],
         [ 94,  84,  76],
         [ 94,  84,  76]],

        ...,

        [[125, 108,  98],
         [161, 144, 134],
         [137, 120, 112],
         ...,
         [ 68,  34,  45],
         [ 72,  39,  48],
         [ 75,  42,  48]],

        [[121, 105,  94],
         [136, 119, 111],
         [ 85,  68,  60],
         ...,
         [ 84,  48,  65],
         [ 85,  51,  65],
         [ 86,  53,  64]],

        [[121, 104,  96],
         [113,  96,  88],
         [ 77,  59

In [97]:
img, label = process_image("images\\cat\\20 Reasons Why Cats Make the Best Pets....webp")
img.numpy()[:2]

array([[[157.      , 186.      , 217.      ],
        [158.      , 187.      , 218.      ],
        [160.      , 189.      , 220.      ],
        [161.      , 190.      , 221.      ],
        [162.      , 191.      , 223.      ],
        [162.      , 191.      , 223.      ],
        [165.      , 194.      , 225.      ],
        [167.      , 194.      , 226.      ],
        [167.      , 195.      , 222.      ],
        [170.      , 195.      , 223.      ],
        [171.      , 198.      , 228.      ],
        [171.      , 198.      , 226.      ],
        [172.      , 200.      , 225.      ],
        [172.      , 200.      , 227.      ],
        [175.      , 200.      , 228.      ],
        [178.      , 200.      , 229.      ],
        [176.      , 201.      , 229.      ],
        [177.      , 202.      , 230.      ],
        [178.      , 203.      , 231.      ],
        [180.      , 205.      , 233.      ],
        [182.      , 205.      , 232.      ],
        [181.      , 205.      , 2

In [98]:
train_ds = train_ds.map(process_image)
test_ds = test_ds.map(process_image)

In [99]:
for image, label in train_ds.take(1):
    print("****",image)
    print("****",label)

**** tf.Tensor(
[[[255. 255. 255.]
  [255. 255. 255.]
  [255. 255. 255.]
  ...
  [255. 255. 255.]
  [255. 255. 255.]
  [255. 255. 255.]]

 [[255. 255. 255.]
  [255. 255. 255.]
  [255. 255. 255.]
  ...
  [255. 255. 255.]
  [255. 255. 255.]
  [255. 255. 255.]]

 [[255. 255. 255.]
  [255. 255. 255.]
  [255. 255. 255.]
  ...
  [255. 255. 255.]
  [255. 255. 255.]
  [255. 255. 255.]]

 ...

 [[255. 255. 255.]
  [255. 255. 255.]
  [255. 255. 255.]
  ...
  [255. 255. 255.]
  [255. 255. 255.]
  [255. 255. 255.]]

 [[255. 255. 255.]
  [255. 255. 255.]
  [255. 255. 255.]
  ...
  [255. 255. 255.]
  [255. 255. 255.]
  [255. 255. 255.]]

 [[255. 255. 255.]
  [255. 255. 255.]
  [255. 255. 255.]
  ...
  [255. 255. 255.]
  [255. 255. 255.]
  [255. 255. 255.]]], shape=(128, 128, 3), dtype=float32)
**** tf.Tensor(b'dog', shape=(), dtype=string)


In [100]:
def scale(image, label):
    return image/255, label

In [101]:
train_ds = train_ds.map(scale)

In [102]:
for image, label in train_ds.take(5):
    print("****Image: ",image.numpy()[0][0])
    print("****Label: ",label.numpy())
    print()

****Image:  [0.24705882 0.27058825 0.3529412 ]
****Label:  b'dog'

****Image:  [0.91064644 0.9184896  0.87128645]
****Label:  b'dog'

****Image:  [0.9098039  0.90588236 0.91764706]
****Label:  b'cat'

****Image:  [0.5106618  0.5756127  0.25882354]
****Label:  b'cat'

****Image:  [0.42745098 0.5019608  0.54901963]
****Label:  b'dog'

